In [2]:
import os
import json
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from supabase import create_client, Client

# 1. 환경 변수 및 클라이언트 설정
load_dotenv()
url = os.environ.get("SUPABASE_URL")
key = os.environ.get("SUPABASE_KEY") # service_role 권장

supabase: Client = create_client(url, key)

def run_supabase_ai_reasoning():
    # 2. AI 모델 및 인스트럭션 설정 (노트북 참조)
    llm = ChatOpenAI(model='gpt-4o', temperature=0.1)
    
    system_instruction = """
    너는 ESG 데이터의 정합성을 최종 진단하고 현장에 명확한 가이드를 내리는 'ESG 실무 데이터 감사인'이다. 
    딱딱한 시스템 용어(L1, L2, L3)를 지우고, 데이터의 성격에 맞는 '현장용 비즈니스 언어'로 재구성하라.

    [실무 용어 치환 매핑]
    - L1 (Z-score/YoY) -> '과거 운영 기록 대비 변동'
    - L2 (Physical Limit) -> '설비 가동 한계 초과'
    - L3 (Intensity Deviation) -> '에너지 투입 효율 저하'

    [진단 및 해설 원칙]
    1. 사람 중심의 해설: 서술형으로 풀어서 설명할 것.
    2. 직관적 수치 배수: '평소 대비 X배', '상한선 대비 Y% 초과'를 사용하여 심각성을 부각할 것.
    3. Extreme Rule 적용: 임계치의 5배 초과 시 '단위 오기입(kWh ↔ MWh)'으로 단정하여 안내할 것.
    """

    # 3. 진단 대상 가져오기 (PostgreSQL 조인 쿼리)
    # Supabase SDK의 조인 기능을 활용하거나 rpc를 사용할 수 있지만, 여기서는 간단히 필터링된 데이터를 가져옵니다.
    print("🔄 진단 대상 데이터 수집 중...")
    
    # 분석이 필요한 이상치 결과들 (analysis_summary가 비어있는 것)
    outliers_query = supabase.table("outlier_results").select("*, standard_usage(*)").is_("analysis_summary", "null").execute()
    targets = outliers_query.data

    if not targets:
        print("💡 진단할 새로운 데이터가 없습니다.")
        return

    for item in targets:
        o_id = item['id']
        std_data = item['standard_usage']
        site = std_data['site_id']
        date = std_data['reporting_date']
        m_name = std_data['metric_name']
        val = std_data['value']
        layer = item['layer']
        limit = item['threshold']

        # 해당 월의 생산량(Activity) 정보 별도 조회
        activity = supabase.table("activity_data").select("production_qty").eq("site_id", site).eq("reporting_date", date).execute()
        prod = activity.data[0]['production_qty'] if activity.data else "기록 없음"

        # 4. LLM에게 전달할 구체적 팩트 구성
        user_content = f"""
        [진단 대상 데이터]
        - 사업장: {site} ({date})
        - 지표: {m_name}
        - 측정값: {val}
        - 시스템 임계치: {limit}
        - 탐지 계층: {layer}
        - 해당 월 생산량: {prod} Ton
        
        위 정보를 바탕으로 아래 JSON 형식에 맞춰 보고서를 작성하라.
        {{
            "이상치_식별자": "{site}_{date}_{m_name}",
            "위험_등급": "Critical/Major/Warning 중 택1",
            "진단_요약": "현장 담당자용 핵심 메시지",
            "판단_근거_및_해설": "비즈니스 언어로 풀어서 설명",
            "추론_가설": "데이터 오기입 또는 현장 이슈 추정",
            "현장_체크리스트": ["점검항목1", "점검항목2", "점검항목3"]
        }}
        """

        response = llm.invoke([
            SystemMessage(content=system_instruction),
            HumanMessage(content=user_content)
        ])
        
        # 5. 결과 파싱 및 DB 업데이트
        analysis_json_raw = response.content.replace("```json", "").replace("```", "").strip()
        
        # JSON 문자열 그대로 DB에 업데이트
        try:
            supabase.table("outlier_results").update({"analysis_summary": analysis_json_raw}).eq("id", o_id).execute()
            print(f"✅ {site} {date} {m_name} AI 진단 완료 및 적재 성공")
        except Exception as e:
            print(f"❌ {site} 업데이트 실패: {e}")

if __name__ == "__main__":
    run_supabase_ai_reasoning()

🔄 진단 대상 데이터 수집 중...
✅ Site A 2025-07-01 electricity AI 진단 완료 및 적재 성공
✅ Site A 2025-10-01 electricity AI 진단 완료 및 적재 성공
✅ Site B 2025-11-01 lng AI 진단 완료 및 적재 성공
